# 00_create_llm_dataset

this code creates the dataset that will be run in the llm.

i uses the cosine merged that already have job ids job titles and role desc tasks skils 

it follows the monthly aproach from 1 to 14, as in the dev mode it is running a 1k per month.

In [1]:
import numpy as np
import json
from pathlib import Path
from datetime import datetime

# ============================================================
# CONFIG
# ============================================================
DEV_ROOT = Path(
    "/projects/a5u/adu_dev/aisi-economy-index/"
    "aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/cosines/cosine_merged"
)

JSONL_ROOT = Path(
    "/projects/a5u/adu_dev/aisi-economy-index/"
    "aisi_economy_index/store/AISI_demo/stage_1_LLM_summary"
)

STAGE3_ROOT = Path(
    "/projects/a5u/adu_dev/aisi-economy-index/"
    "aisi_economy_index/store/AISI_demo/stage_3/dev/llm_negative_selection"
)

EMBEDS = ["bge_large", "e5_large", "gte_large"]
MONTHS = [f"adzuna_month{m:02d}" for m in range(1, 15)]
OVERWRITE = True


# ============================================================
# HELPERS
# ============================================================
def require_npz_keys(z, need, ctx=""):
    have = set(z.files)
    missing = sorted(list(set(need) - have))
    if missing:
        raise KeyError(
            f"NPZ missing keys {missing} | ctx={ctx} | have={sorted(list(have))}"
        )


def scan_jsonl_dir(jsonl_dir: Path, job_id_set: set):
    title_map = {}
    sector_map = {}
    desc_map = {}
    total_files = 0
    found = 0

    for p in sorted(jsonl_dir.glob("*.jsonl")):
        total_files += 1
        with p.open("r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                except json.JSONDecodeError:
                    continue

                jid = (
                    obj.get("job_id")
                    or obj.get("jobid")
                    or obj.get("id")
                    or (obj.get("meta") or {}).get("job_id")
                )
                if jid is None:
                    continue

                jid = str(jid)
                if jid not in job_id_set:
                    continue

                found += 1

                t = obj.get("job_ad_title") or obj.get("title") or obj.get("job_title") or ""
                c = obj.get("job_sector_category") or obj.get("category_name") or obj.get("category") or ""
                d = (
                    obj.get("job_description")
                    or obj.get("description")
                    or obj.get("full_text")
                    or obj.get("text")
                    or ""
                )

                if t and jid not in title_map:
                    title_map[jid] = t
                if c and jid not in sector_map:
                    sector_map[jid] = c
                if d and jid not in desc_map:
                    desc_map[jid] = d

    return title_map, sector_map, desc_map, total_files, found


def load_stage2_npz(in_npz: Path, ctx: str):
    """
    Accepts either:
      - old schema: short_desc, tasks_and_skills
      - cosine_merged schema: role_text, taskskill_text
    Returns canonical arrays: job_ids, job_desc, job_tasks, domains, titles
    """
    with np.load(in_npz, allow_pickle=True) as z:
        require_npz_keys(z, {"job_ids", "domains", "candidates"}, ctx)

        job_ids = z["job_ids"].astype(str)
        domains = z["domains"].astype(object)
        titles = z["candidates"]

        if "short_desc" in z.files:
            job_desc = z["short_desc"].astype(object)
        elif "role_text" in z.files:
            job_desc = z["role_text"].astype(object)
        else:
            raise KeyError(f"{ctx}: missing short_desc/role_text")

        if "tasks_and_skills" in z.files:
            job_tasks = z["tasks_and_skills"].astype(object)
        elif "taskskill_text" in z.files:
            job_tasks = z["taskskill_text"].astype(object)
        else:
            raise KeyError(f"{ctx}: missing tasks_and_skills/taskskill_text")

    return job_ids, job_desc, job_tasks, domains, titles


# ============================================================
# RUN
# ============================================================
print("START:", datetime.now().isoformat(timespec="seconds"), flush=True)
print("DEV_ROOT:", DEV_ROOT, flush=True)
print("JSONL_ROOT:", JSONL_ROOT, flush=True)
print("STAGE3_ROOT:", STAGE3_ROOT, flush=True)

for embed in EMBEDS:
    print("\n" + "=" * 110, flush=True)
    print("EMBED:", embed, flush=True)

    FINAL_DIR = DEV_ROOT / embed
    OUT_DIR = STAGE3_ROOT / embed
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    print("FINAL_DIR:", FINAL_DIR, flush=True)
    print("OUT_DIR:", OUT_DIR, flush=True)

    for month in MONTHS:
        print("\n------------------------------", flush=True)
        print("MONTH:", month, flush=True)

        in_npz = FINAL_DIR / f"{month}.npz"
        jsonl_dir = JSONL_ROOT / f"{month}_npz"
        out_npz = OUT_DIR / f"{month}.npz"

        if out_npz.exists() and not OVERWRITE:
            print("SKIP exists:", out_npz.name, flush=True)
            continue

        if not in_npz.exists():
            print("MISS stage2 npz:", in_npz, flush=True)
            continue

        if not jsonl_dir.exists():
            print("MISS jsonl dir:", jsonl_dir, flush=True)
            continue

        ctx = f"{embed} {month} {in_npz}"
        job_ids, job_desc, job_tasks, domains, titles = load_stage2_npz(in_npz, ctx)

        n = len(job_ids)
        job_id_set = set(job_ids)

        print("Rows:", n, flush=True)

        # ---- scan jsonl ----
        print("Scanning JSONL dir:", jsonl_dir, flush=True)
        title_map, sector_map, desc_map, total_files, found = scan_jsonl_dir(jsonl_dir, job_id_set)

        print("JSONL files:", total_files, flush=True)
        print("Matched JSONL rows:", found, flush=True)

        # ---- align back to npz order ----
        job_ad_title = np.empty(n, dtype=object)
        job_sector_category = np.empty(n, dtype=object)
        job_description = np.empty(n, dtype=object)

        miss_meta = 0
        miss_desc = 0

        for i, jid in enumerate(job_ids):
            t = title_map.get(jid, "")
            c = sector_map.get(jid, "")
            d = desc_map.get(jid, "")

            if not t and not c:
                miss_meta += 1
            if not d:
                miss_desc += 1

            job_ad_title[i] = t
            job_sector_category[i] = c
            job_description[i] = d

        # ---- write NPZ for LLM runner (canonical keys) ----
        np.savez_compressed(
            out_npz,
            job_ids=job_ids.astype(object),
            job_desc=job_desc,
            job_tasks=job_tasks,
            domain=domains,
            titles=titles,
            job_ad_title=job_ad_title,
            job_sector_category=job_sector_category,
            job_description=job_description,
        )

        print(
            f"OK wrote {out_npz} rows={n:,} "
            f"missing_title_sector={miss_meta:,} missing_full_desc={miss_desc:,}",
            flush=True,
        )

print("\nEND:", datetime.now().isoformat(timespec="seconds"), flush=True)


START: 2026-02-05T15:51:14
DEV_ROOT: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/cosines/cosine_merged
JSONL_ROOT: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_1_LLM_summary
STAGE3_ROOT: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/dev/llm_negative_selection

EMBED: bge_large
FINAL_DIR: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_2_embeddings_and_cosines/dev/cosines/cosine_merged/bge_large
OUT_DIR: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_3/dev/llm_negative_selection/bge_large

------------------------------
MONTH: adzuna_month01
Rows: 1000
Scanning JSONL dir: /projects/a5u/adu_dev/aisi-economy-index/aisi_economy_index/store/AISI_demo/stage_1_LLM_summary/adzuna_month01_npz
JSONL files: 100
Matched JSONL rows: 1000
OK wrote /projects/a5u/adu_dev/aisi-economy-index/aisi